# 03 - Models: XGBoost + Isolation Forest

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

Trains the mule-detection models on the point-in-time features from notebook 02, compares them with the rules baseline, and saves everything the live scoring service and demo need.

- **XGBoost** learns from labelled laundering (supervised).
- **Isolation Forest** flags payments that are simply unusual, no labels needed (unsupervised).
- Combined, they make the **multi-model** system the guidebook asks for.

In [ ]:
import os, json, time, platform
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
try:
    import xgboost as xgb
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb
import joblib

SHARED = os.path.expanduser('~/shared')
MODELS = f'{SHARED}/models'
os.makedirs(MODELS, exist_ok=True)
print('Machine:', platform.machine(), '| xgboost', xgb.__version__)

data = pd.read_pickle(f'{SHARED}/features.pkl')
meta = json.load(open(f'{SHARED}/feature_list.json'))
FEATURES = meta['features']
print(data.shape, '|', len(FEATURES), 'features')

In [ ]:
def evaluate(y, score, rate=0.01):
    y = np.asarray(y); score = np.asarray(score, dtype='float64')
    k = max(1, int(len(y) * rate))
    top = np.argsort(-score, kind='stable')[:k]
    return {
        'recall@1%': y[top].sum() / max(y.sum(), 1),
        'precision@1%': y[top].mean(),
        'pr_auc': average_precision_score(y, score),
    }

def rank01(x):
    """Turn any score into a 0-1 rank so different models can be combined fairly."""
    return pd.Series(x).rank(pct=True).values

train = data[data['split'] == 'train']
val = data[data['split'] == 'val']
test = data[data['split'] == 'test']
X_tr, y_tr = train[FEATURES], train['y']
X_va, y_va = val[FEATURES], val['y']
X_te, y_te = test[FEATURES], test['y']
for name, d in [('train', train), ('val', val), ('test', test)]:
    print(f"{name:5s} {len(d):>9,} rows | {int(d['y'].sum()):>5,} laundering")

## 1. XGBoost
- `scale_pos_weight` makes each rare laundering example count as much as ~1,000 normal ones, so the model doesn't just predict "not laundering" every time.
- It trains on 2–6 Sep and stops early when the validation score (7–8 Sep) stops improving, so it doesn't memorise the training days.

In [ ]:
spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
clf = xgb.XGBClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    scale_pos_weight=spw, tree_method='hist', eval_metric='aucpr',
    early_stopping_rounds=40, n_jobs=-1, random_state=42,
)
t0 = time.time()
clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=50)
print(f'Trained on {platform.machine()} in {time.time() - t0:.1f}s | best iteration {clf.best_iteration} | scale_pos_weight {spw:.0f}')
booster = clf.get_booster()[: clf.best_iteration + 1]   # keep only the trees up to the early-stopping point

p_va = clf.predict_proba(X_va)[:, 1]
pd.Series(evaluate(y_va, p_va)).round(4)

## 2. Isolation Forest
Learns what "normal" payments look like from a sample of training data, then scores how unusual each payment is. It never sees the laundering labels.

In [ ]:
t0 = time.time()
sample = X_tr.sample(n=min(300_000, len(X_tr)), random_state=42)
iso = IsolationForest(n_estimators=200, max_samples=256, random_state=42, n_jobs=-1).fit(sample)
a_va = -iso.score_samples(X_va)   # higher = more unusual
print(f'Isolation Forest trained + scored in {time.time() - t0:.1f}s')
pd.Series(evaluate(y_va, a_va)).round(4)

## 3. Combine the two
We blend the two models' rankings: `combined = (1 - w) * XGBoost + w * IsolationForest`.
The weight `w` is chosen on **validation** days only. The test days stay untouched until the final score.

In [ ]:
r_p, r_a = rank01(p_va), rank01(a_va)
weights = [0.0, 0.05, 0.1, 0.2, 0.3]
grid = {w: evaluate(y_va, (1 - w) * r_p + w * r_a)['recall@1%'] for w in weights}
W = max(grid, key=grid.get)
print('Validation recall@1% by weight:', {k: round(float(v), 4) for k, v in grid.items()})
print('Chosen weight for Isolation Forest:', W)

# Alert threshold = the score that flags the top 1% of validation payments
comb_va = (1 - W) * r_p + W * r_a
THRESHOLD = float(np.quantile(comb_va, 0.99))

## 4. Final test: rules vs XGBoost vs combined (9–10 Sep, never seen before)

In [ ]:
ach = int(test['payment_format'][test['payment_format_name'] == 'ACH'].iloc[0]) if (test['payment_format_name'] == 'ACH').any() else -1
def rules_score(d):
    flags = ((d['payment_format'] == ach).astype(int) + d['new_pair']
             + (d['s_new_receivers_24h'] >= 3).astype(int) + (d['r_new_senders_24h'] >= 2).astype(int)
             + (d['r_age_h'] < 24).astype(int))
    return flags + 1e-3 * d['log_amount']

p_te = clf.predict_proba(X_te)[:, 1]
a_te = -iso.score_samples(X_te)
comb_te = (1 - W) * rank01(p_te) + W * rank01(a_te)

results = pd.DataFrame({
    'Rules baseline': evaluate(y_te, rules_score(test)),
    'Isolation Forest only': evaluate(y_te, a_te),
    'XGBoost': evaluate(y_te, p_te),
    f'XGBoost + IsolationForest (w={W})': evaluate(y_te, comb_te),
}).T.round(4)
results

## 5. "Isn't it just detecting ACH?"
Judges may ask this, since ACH carries most of the laundering. We re-score **only ACH payments**: if the model still beats the rules there, it has learned real mule behaviour, not just payment type.

In [ ]:
is_ach = (test['payment_format_name'] == 'ACH').values
ach_results = pd.DataFrame({
    'Rules baseline (ACH only)': evaluate(y_te[is_ach], rules_score(test[is_ach])),
    'Combined model (ACH only)': evaluate(y_te[is_ach], comb_te[is_ach]),
}).T.round(4)
print(f"ACH test payments: {is_ach.sum():,} | laundering among them: {int(y_te[is_ach].sum()):,}")
ach_results

## 6. What the model relies on
Feature importance (gain): how much each feature improves the model's decisions. Useful for the pitch slide and sanity checking.

In [ ]:
imp = pd.Series(booster.get_score(importance_type='gain')).reindex(FEATURES).fillna(0)
(imp / imp.sum()).sort_values(ascending=False).round(3).head(12)

## 7. Top reasons per alert (feeds the LLM)
XGBoost can say exactly how much each feature pushed a payment's score up (SHAP contributions). We turn the top 3 into plain-English reasons. The LLM later writes the student warning and analyst explanation **only from these reasons**, so it can't invent anything.

In [ ]:
DESCRIBE = {
    'log_amount':           lambda v, r: f"amount of {np.expm1(v):,.0f}",
    'payment_format':       lambda v, r: f"paid by {r['payment_format_name']}",
    'payment_currency':     lambda v, r: "payment currency is unusual for laundering patterns",
    'cross_currency':       lambda v, r: "payment converted between currencies" if v else "same-currency payment",
    'cross_bank':           lambda v, r: "sent from a different bank" if v else "sent within the same bank",
    'hour':                 lambda v, r: f"sent at {int(v):02d}:00",
    'new_pair':             lambda v, r: "first ever payment between these two accounts" if v else "these accounts have transacted before",
    'pair_prior_cnt':       lambda v, r: f"{int(v)} earlier payments between these two accounts",
    'r_in_cnt_all':         lambda v, r: f"receiving account has had {int(v)} incoming payments in total",
    'r_in_cnt_24h':         lambda v, r: f"receiving account got {int(v)} payments in the last 24h",
    'r_in_cnt_1h':          lambda v, r: f"receiving account got {int(v)} payments in the last hour",
    'r_out_cnt_all':        lambda v, r: f"receiving account has sent {int(v)} payments in total",
    'r_out_cnt_24h':        lambda v, r: f"receiving account sent {int(v)} payments out in the last 24h",
    'r_distinct_senders':   lambda v, r: f"receiving account has been paid by {int(v)} different senders",
    'r_new_senders_24h':    lambda v, r: f"receiving account was paid by {int(v)} new senders in the last 24h",
    'amt_vs_r_avg':         lambda v, r: "first payment this account has received" if v < 0 else f"{v:.1f}x this account's usual incoming amount",
    'r_passthrough_24h':    lambda v, r: f"{v:.0%} of the last 24h's incoming money has already been sent on",
    's_out_cnt_all':        lambda v, r: f"sender has made {int(v)} payments in total",
    's_out_cnt_24h':        lambda v, r: f"sender made {int(v)} payments in the last 24h",
    's_in_cnt_24h':         lambda v, r: f"sender received {int(v)} payments in the last 24h",
    's_distinct_receivers': lambda v, r: f"sender has paid {int(v)} different accounts",
    's_new_receivers_24h':  lambda v, r: f"sender paid {int(v)} new accounts in the last 24h",
    'r_age_h':              lambda v, r: f"receiving account first seen {v:.0f} hours ago",
    's_age_h':              lambda v, r: f"sender first seen {v:.0f} hours ago",
}

def top_reasons(rows, k=3):
    contrib = booster.predict(xgb.DMatrix(rows[FEATURES]), pred_contribs=True)[:, :-1]
    out = []
    for i in range(len(rows)):
        row = rows.iloc[i]
        idx = np.argsort(-contrib[i])[:k]
        out.append([DESCRIBE[FEATURES[j]](row[FEATURES[j]], row) for j in idx if contrib[i, j] > 0])
    return out

# Example: the 5 riskiest test payments and why
top5 = test.assign(score=comb_te).nlargest(5, 'score')
for (_, row), reasons in zip(top5.iterrows(), top_reasons(top5)):
    print(f"{row['timestamp']}  {row['sender']} -> {row['receiver']}  score={row['score']:.3f}  laundering={row['y']}")
    for reason in reasons:
        print('   -', reason)

## 8. Save models + demo data
- `models/` holds everything the scoring service needs.
- `demo_alerts.json` gives the front-end team real flagged payments with reasons to build against straight away.

In [ ]:
booster.save_model(f'{MODELS}/mule_xgb.json')
joblib.dump(iso, f'{MODELS}/iso_forest.joblib')
# Reference distributions so the service can turn a single payment's raw scores into 0-1 ranks
np.save(f'{MODELS}/ref_xgb_val.npy', np.sort(p_va))
np.save(f'{MODELS}/ref_iso_val.npy', np.sort(a_va))
config = {
    'features': FEATURES,
    'iso_weight': W,
    'alert_threshold': THRESHOLD,
    'scale_pos_weight': spw,
    'best_iteration': int(clf.best_iteration),
    'test_results': results.to_dict(orient='index'),
    'test_results_ach_only': ach_results.to_dict(orient='index'),
}
json.dump(config, open(f'{MODELS}/config.json', 'w'), indent=2, default=float)

# Demo alerts: top flagged test payments (mix of true and false alarms, like real life)
demo = test.assign(score=comb_te).nlargest(200, 'score')
demo_records = []
for (_, row), reasons in zip(demo.iterrows(), top_reasons(demo)):
    demo_records.append({
        'timestamp': str(row['timestamp']), 'sender': row['sender'], 'receiver': row['receiver'],
        'amount': float(row['amount_paid']), 'payment_format': row['payment_format_name'],
        'risk_score': round(float(row['score']), 4), 'is_laundering': int(row['y']),
        'reasons': reasons,
    })
json.dump(demo_records, open(f'{SHARED}/demo_alerts.json', 'w'), indent=2)
print('Saved:', sorted(os.listdir(MODELS)), '+ shared/demo_alerts.json')